In [1]:
import pandas as pd

data_raw = [
    ("Quero devolver este sofa que chegou com rasgo", "trocas_devolucoes"),
    ("Gostaria de trocar minha mesa veio arranhada", "trocas_devolucoes"),
    ("Como faco para solicitar a devolucao do meu estofado?", "trocas_devolucoes"),
    ("O rack veio com defeito e quero trocar", "trocas_devolucoes"),
    ("Preciso devolver a cadeira de escritorio com defeito", "trocas_devolucoes"),
    ("Quero cancelar a compra e pedir extorno do sofá", "trocas_devolucoes"),
    ("Viu meu painel de tv veio quebrado quero troca", "trocas_devolucoes"),
    ("Gostaria de devolver o armário por defeito", "trocas_devolucoes"),

    ("Qual o status da entrega da minha estante?", "logistica_entregas"),
    ("Onde esta meu pedido de poltrona?", "logistica_entregas"),
    ("Qual o prazo de entrega do sofa que comprei?", "logistica_entregas"),
    ("Meu armario de cozinha ainda nao chegou", "logistica_entregas"),
    ("Quero rastrear o transporte da minha mesa de jantar", "logistica_entregas"),
    ("A entrega do guarda roupa esta atrasada", "logistica_entregas"),
    ("Quando chega minha cadeira presidente?", "logistica_entregas"),
    ("Saber dia que chegam meus moveis", "logistica_entregas"),

    ("Como montar o painel de tv da sala?", "suporte_tecnico"),
    ("Nao consigo entender o manual de montagem do rack", "suporte_tecnico"),
    ("Faltaram parafusos no kit do meu guarda roupa", "suporte_tecnico"),
    ("Preciso de ajuda para ajustar a porta do armario", "suporte_tecnico"),
    ("A peca B da mesa nao encaixa na peca C", "suporte_tecnico"),
    ("Como regulo a altura da minha cadeira ergonomica?", "suporte_tecnico"),
    ("Voces enviam montador para a estante?", "suporte_tecnico"),
    ("Manual da cama de casal veio em branco", "suporte_tecnico"),

    ("Qual o valor da mesa de jantar 6 lugares?", "vendas_orcamento"),
    ("Gostaria de um orcamento de sofa retratil", "vendas_orcamento"),
    ("Voces tem desconto para pagamento via pix na poltrona?", "vendas_orcamento"),
    ("Quanto custa o frete para o guarda roupa de casal?", "vendas_orcamento"),
    ("Tem promocao de comoda este mes?", "vendas_orcamento"),
    ("Qual o preço do armario de cozinha planejado?", "vendas_orcamento"),
    ("Gostaria de comprar um beliche de madeira", "vendas_orcamento"),
    ("Quais as formas de parcelamento do rack?", "vendas_orcamento")
]

df = pd.DataFrame(data_raw, columns=["mensagem", "intencao"])
df.to_csv("sac_moveis_ac2.csv", index=False)

print("Dataset 'sac_moveis_ac2.csv' gerado com sucesso!")

Dataset 'sac_moveis_ac2.csv' gerado com sucesso!


In [3]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 90.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import re
import nltk
from nltk.corpus import stopwords
import spacy

nltk.download("stopwords", quiet=True)
stop_words_pt = set(stopwords.words("portuguese"))

nlp = spacy.load("pt_core_news_sm")

def limpar_e_lemmatizar(texto):
    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens_filtrados = [
        token.lemma_
        for token in doc
        if token.text not in stop_words_pt
        and not token.is_space
        and len(token.text) > 1
    ]

    return " ".join(tokens_filtrados)

In [6]:
frase_teste = "Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!"

print("Frase Original:", frase_teste)
print("Frase Limpa & Lemmatizada:", limpar_e_lemmatizar(frase_teste))

Frase Original: Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!
Frase Limpa & Lemmatizada: gostar saber devolver valor meso comprada


In [8]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 60.6 MB/s eta 0:00:00


In [11]:
import numpy as np
import pandas as pd
import gensim.downloader as api

print("Carregando modelo de Embeddings FastText (Gensim)...")

fasttext_model = api.load("glove-wiki-gigaword-50")

def obter_vetor_frase(frase, model):
    palavras = frase.split()
    vetores = []

    for palavra in palavras:
        if palavra in model:
            vetores.append(model[palavra])

    if len(vetores) == 0:
        return np.zeros(model.vector_size)

    vetor_medio = np.mean(vetores, axis=0)

    return vetor_medio


    # utilizamos o Mean Pooling pra fazer o calculo da média dos vetores das palavras de cada frase.
    # a mensagem foi representada por um único vetor. a matriz que gerou possui 32 exemplos e 50 dimensões.

Carregando modelo de Embeddings FastText (Gensim)...


In [12]:
df = pd.read_csv("sac_moveis_ac2.csv")

X_vetores = np.array([
    obter_vetor_frase(msg, fasttext_model)
    for msg in df["mensagem"]
])

print(
    "Formato da Matriz de Vetores Densos (Exemplos, Dimensões):",
    X_vetores.shape
)

Formato da Matriz de Vetores Densos (Exemplos, Dimensões): (32, 50)


In [13]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv("sac_moveis_ac2.csv")

y = df["intencao"]

modelo_regressao = LogisticRegression(max_iter=1000)

modelo_regressao.fit(
    X_vetores,
    y
)

def classificar_com_fallback_linear(
    mensagem_usuario,
    modelo,
    model_emb,
    limiar=0.50
):
    vetor_msg = obter_vetor_frase(
        mensagem_usuario,
        model_emb
    ).reshape(1, -1)

    probabilidades = modelo.predict_proba(vetor_msg)[0]

    max_prob = np.max(probabilidades)

    idx_classe = np.argmax(probabilidades)

    intencao_prevista = modelo.classes_[idx_classe]

    if max_prob < limiar:
        return "FALLBACK_HUMANO", max_prob
    else:
        return intencao_prevista, max_prob

In [14]:
testes = [
    "Quero saber o valor do frete do sofá",
    "Gostaria de ver receitas de bolo de cenoura"
]

for t in testes:
    intencao, conf = classificar_com_fallback_linear(
        t,
        modelo_regressao,
        fasttext_model
    )

    print(
        f"Frase: '{t}' | Resultado: {intencao} | Confiança: {conf:.2%}"
    )

Frase: 'Quero saber o valor do frete do sofá' | Resultado: FALLBACK_HUMANO | Confiança: 35.52%
Frase: 'Gostaria de ver receitas de bolo de cenoura' | Resultado: vendas_orcamento | Confiança: 69.10%


In [15]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

modelo_knn = KNeighborsClassifier(n_neighbors=3)

modelo_knn.fit(
    X_vetores,
    y
)

y_pred_linear = modelo_regressao.predict(
    X_vetores
)

y_pred_knn = modelo_knn.predict(
    X_vetores
)

acuracia_linear = accuracy_score(
    y,
    y_pred_linear
)

acuracia_knn = accuracy_score(
    y,
    y_pred_knn
)

print(
    f"Acurácia - Regressão Logística (Linear): {acuracia_linear:.2%}"
)

print(
    f"Acurácia - KNN (Distância K=3): {acuracia_knn:.2%}"
)

Acurácia - Regressão Logística (Linear): 93.75%
Acurácia - KNN (Distância K=3): 56.25%
